# RAG Injection Pipeline — ArchGuard-AI

Pipeline de processamento dos documentos STRIDE para ingestão em banco vetorial.

**Fonte:** `RAG/*.md` (12 documentos)  
**Embedding Model:** `nomic-embed-text-v1.5` via LM Studio  
**Vector Store:** ChromaDB  
**Estratégia:** Chunking por seções H2, metadados extraídos do blockquote

In [1]:
"""
=============================================================
CELL 1: Imports e Paths
=============================================================
"""
import os
import re
import json
from pathlib import Path
from typing import Optional

PROJECT_ROOT = Path(os.getcwd()).parent
RAG_DIR = PROJECT_ROOT / "RAG"

md_files = sorted([
    f for f in RAG_DIR.glob("*.md")
    if f.name != "README.md"
])

print(f"✅ RAG dir: {RAG_DIR}")
print(f"   Documentos encontrados: {len(md_files)}")
for f in md_files:
    print(f"   • {f.name}")

✅ RAG dir: /Users/pedromarquardt/dev/estudos/tech-challenger/archguard-ai/ArchGuard-AI/RAG
   Documentos encontrados: 12
   • 01-stride-methodology-overview.md
   • 02-stride-llm-applications-arxiv-2406.11007.md
   • 03-stride-ai-framework-arxiv-2605.17163.md
   • 04-llms-stride-network-security-arxiv-2505.04101.md
   • 05-stride-cicd-supply-chain-arxiv-2506.06478.md
   • 06-stride-cloud-security-cisa-aligned.md
   • 07-astride-agentic-ai-arxiv-2512.04785.md
   • 08-stride-microservices-distributed-architectures.md
   • 09-isadm-stride-attack-d3fend-arxiv-2512.18751.md
   • 10-threatmodeling-llm-banking-arxiv-2411.17058.md
   • 11-aegisshield-democratizing-threat-modeling-arxiv-2509.10482.md
   • 12-domain-adapted-llm-stride-empirical-arxiv-2605.10808.md


In [21]:
"""
=============================================================
CELL 2: Extração de Metadados do Blockquote
=============================================================
"""

def extract_metadata(content: str) -> dict:
    """
    Extrai metadados do blockquote no início do documento.
    Formato esperado: > **Campo:** Valor
    """
    metadata = {}
    
    # Regex para capturar linhas do blockquote com formato **Key:** Value
    # Nota: nos docs o formato é **Campo:** (colon dentro do bold)
    pattern = re.compile(r'>\s*\*\*(.+?):?\*\*:?\s+(.+)')
    
    for match in pattern.finditer(content):
        key = match.group(1).strip().rstrip(":").lower()
        value = match.group(2).strip()
        
        # Normaliza as chaves — inclui variações entre documentos
        key_map = {
            "fonte": "fonte",
            "autores": "autores",
            "ano": "ano",
            "url": "url",
            "url cisa": "url",
            "categoria arxiv": "categoria_arxiv",
            "tipo": "tipo",
            "relevância para o projeto": "relevancia",
            "publicações base": "publicacoes_base",
        }
        
        normalized_key = key_map.get(key, key.replace(" ", "_"))
        metadata[normalized_key] = value
    
    return metadata


def extract_title(content: str) -> str:
    """Extrai o título H1 do documento."""
    match = re.match(r'^#\s+(.+)', content)
    return match.group(1).strip() if match else "Sem título"


# Teste com o primeiro documento
test_content = md_files[0].read_text(encoding="utf-8")
test_metadata = extract_metadata(test_content)
test_title = extract_title(test_content)

print(f"📄 Teste: {md_files[0].name}")
print(f"   Título: {test_title}")
print(f"   Metadados extraídos:")
for k, v in test_metadata.items():
    print(f"     {k}: {v}")

# Verificar chaves únicas em todos os docs
print(f"\n📊 Chaves de metadados em todos os docs:")
all_keys = set()
for f in md_files:
    meta = extract_metadata(f.read_text(encoding="utf-8"))
    all_keys.update(meta.keys())
print(f"   {sorted(all_keys)}")

📄 Teste: 01-stride-methodology-overview.md
   Título: STRIDE Threat Modeling Methodology — Overview Completo
   Metadados extraídos:
     fonte: Microsoft (criação original), OWASP, Threat Modeling Manifesto
     tipo: Revisão conceitual / referência metodológica
     relevancia: Base conceitual para o motor de análise do ArchGuard-AI

📊 Chaves de metadados em todos os docs:
   ['ano', 'autor', 'autores', 'categoria_arxiv', 'fonte', 'publicacoes_base', 'referências', 'relevancia', 'tipo', 'url']


In [25]:
"""
=============================================================
CELL 3: Lógica de Chunking por Seções
=============================================================
"""

MAX_CHUNK_CHARS = 6000  # ~1500 tokens
MIN_CHUNK_CHARS = 100


def split_by_headers(content: str, level: str = "## ") -> list[tuple[str, str]]:
    """
    Divide o conteúdo por headers de um nível específico.
    Retorna lista de (título_da_seção, conteúdo_da_seção).
    """
    sections = []
    current_title = ""
    current_lines = []
    
    for line in content.split("\n"):
        if line.startswith(level) and not line.startswith(level + "#"):
            # Salva seção anterior
            if current_lines:
                sections.append((current_title, "\n".join(current_lines).strip()))
            current_title = line.lstrip("#").strip()
            current_lines = [line]
        else:
            current_lines.append(line)
    
    # Última seção
    if current_lines:
        sections.append((current_title, "\n".join(current_lines).strip()))
    
    return sections


def remove_frontmatter(content: str) -> str:
    """
    Remove o blockquote de metadados e o primeiro separador ---.
    Retorna o conteúdo a partir da primeira seção ##.
    """
    lines = content.split("\n")
    body_start = 0
    
    # Pula título H1, blockquote e separador
    found_separator = False
    for i, line in enumerate(lines):
        if line.strip() == "---" and i > 0:
            if not found_separator:
                found_separator = True
                body_start = i + 1
                break
    
    return "\n".join(lines[body_start:]).strip()


def chunk_document(filepath: Path) -> list[dict]:
    """
    Processa um documento MD e retorna chunks com metadados.
    """
    content = filepath.read_text(encoding="utf-8")
    
    # Extrair metadados e título
    metadata = extract_metadata(content)
    title = extract_title(content)
    
    # Remover frontmatter (blockquote + separador)
    body = remove_frontmatter(content)
    
    # Split por H2
    h2_sections = split_by_headers(body, "## ")
    
    chunks = []
    doc_id = filepath.stem.split("-")[0]  # "01", "02", etc.
    section_idx = 0
    
    for section_title, section_content in h2_sections:
        # Ignorar seções vazias ou apenas separadores
        clean = section_content.replace("---", "").strip()
        if len(clean) < MIN_CHUNK_CHARS:
            continue
        
        section_idx += 1
        
        # Se seção > MAX_CHUNK_CHARS, subdividir por H3
        if len(section_content) > MAX_CHUNK_CHARS:
            h3_sections = split_by_headers(section_content, "### ")
            
            for sub_idx, (sub_title, sub_content) in enumerate(h3_sections):
                if len(sub_content.strip()) < MIN_CHUNK_CHARS:
                    continue
                
                # Prefixo de contexto hierárquico
                chunk_text = f"# {title}\n## {section_title}\n### {sub_title}\n\n{sub_content}" if sub_title else f"# {title}\n## {section_title}\n\n{sub_content}"
                
                chunks.append({
                    "id": f"doc_{doc_id}_s{section_idx:02d}_sub{sub_idx:02d}",
                    "text": chunk_text,
                    "metadata": {
                        **metadata,
                        "source_file": filepath.name,
                        "title": title,
                        "section_title": section_title,
                        "subsection_title": sub_title or None,
                        "section_index": section_idx,
                    }
                })
        else:
            # Chunk inteiro com prefixo de contexto
            chunk_text = f"# {title}\n## {section_title}\n\n{section_content}"
            
            chunks.append({
                "id": f"doc_{doc_id}_s{section_idx:02d}",
                "text": chunk_text,
                "metadata": {
                    **metadata,
                    "source_file": filepath.name,
                    "title": title,
                    "section_title": section_title,
                    "subsection_title": None,
                    "section_index": section_idx,
                }
            })
    
    return chunks


# Teste com primeiro doc
test_chunks = chunk_document(md_files[0])
print(f"📄 {md_files[0].name}")
print(f"   Chunks gerados: {len(test_chunks)}")
print(f"\n   Exemplo (primeiro chunk):")
print(f"   ID: {test_chunks[0]['id']}")
print(f"   Seção: {test_chunks[0]['metadata']['section_title']}")
print(f"   Tamanho: {len(test_chunks[0]['text'])} chars")
print(f"   Texto (primeiros 200 chars):\n   {test_chunks[0]['text'][:200]}...")

📄 01-stride-methodology-overview.md
   Chunks gerados: 7

   Exemplo (primeiro chunk):
   ID: doc_01_s01
   Seção: 1. O Que é STRIDE?
   Tamanho: 1254 chars
   Texto (primeiros 200 chars):
   # STRIDE Threat Modeling Methodology — Overview Completo
## 1. O Que é STRIDE?

## 1. O Que é STRIDE?

STRIDE é uma metodologia de *threat modeling* (modelagem de ameaças) criada pela Microsoft no fin...


In [26]:
"""
=============================================================
CELL 4: Processar Todos os Documentos
=============================================================
"""

all_chunks = []

for md_file in md_files:
    chunks = chunk_document(md_file)
    all_chunks.extend(chunks)
    print(f"✅ {md_file.name:55s} → {len(chunks):3d} chunks")

print(f"\n{'='*60}")
print(f"📊 Total de chunks gerados: {len(all_chunks)}")
print(f"   Documentos processados:  {len(md_files)}")

✅ 01-stride-methodology-overview.md                       →   7 chunks
✅ 02-stride-llm-applications-arxiv-2406.11007.md          →   8 chunks
✅ 03-stride-ai-framework-arxiv-2605.17163.md              →   8 chunks
✅ 04-llms-stride-network-security-arxiv-2505.04101.md     →   9 chunks
✅ 05-stride-cicd-supply-chain-arxiv-2506.06478.md         →   9 chunks
✅ 06-stride-cloud-security-cisa-aligned.md                →  12 chunks
✅ 07-astride-agentic-ai-arxiv-2512.04785.md               →   8 chunks
✅ 08-stride-microservices-distributed-architectures.md    →  12 chunks
✅ 09-isadm-stride-attack-d3fend-arxiv-2512.18751.md       →   9 chunks
✅ 10-threatmodeling-llm-banking-arxiv-2411.17058.md       →   8 chunks
✅ 11-aegisshield-democratizing-threat-modeling-arxiv-2509.10482.md →  11 chunks
✅ 12-domain-adapted-llm-stride-empirical-arxiv-2605.10808.md →   8 chunks

📊 Total de chunks gerados: 109
   Documentos processados:  12


In [27]:
"""
=============================================================
CELL 5: Verificação e Estatísticas
=============================================================
"""

# Distribuição de tamanho dos chunks
sizes = [len(c["text"]) for c in all_chunks]
token_estimates = [s // 4 for s in sizes]  # estimativa ~4 chars/token

print(f"📈 Estatísticas dos Chunks:")
print(f"   Total:          {len(all_chunks)}")
print(f"   Menor chunk:    {min(sizes):,} chars (~{min(token_estimates):,} tokens)")
print(f"   Maior chunk:    {max(sizes):,} chars (~{max(token_estimates):,} tokens)")
print(f"   Média:          {sum(sizes)//len(sizes):,} chars (~{sum(token_estimates)//len(token_estimates):,} tokens)")
print(f"   Chunks > 8192 tokens: {sum(1 for t in token_estimates if t > 8192)}")

# Verificar metadados completos
missing_metadata = []
for chunk in all_chunks:
    if not chunk["metadata"].get("title"):
        missing_metadata.append(chunk["id"])

if missing_metadata:
    print(f"\n⚠️  Chunks sem título: {missing_metadata}")
else:
    print(f"\n✅ Todos os chunks têm metadados completos")

# Distribuição por documento
print(f"\n📊 Chunks por documento:")
from collections import Counter
doc_counts = Counter(c["metadata"]["source_file"] for c in all_chunks)
for doc, count in sorted(doc_counts.items()):
    print(f"   {doc:55s} → {count:3d}")

📈 Estatísticas dos Chunks:
   Total:          109
   Menor chunk:    405 chars (~101 tokens)
   Maior chunk:    4,220 chars (~1,055 tokens)
   Média:          1,235 chars (~308 tokens)
   Chunks > 8192 tokens: 0

✅ Todos os chunks têm metadados completos

📊 Chunks por documento:
   01-stride-methodology-overview.md                       →   7
   02-stride-llm-applications-arxiv-2406.11007.md          →   8
   03-stride-ai-framework-arxiv-2605.17163.md              →   8
   04-llms-stride-network-security-arxiv-2505.04101.md     →   9
   05-stride-cicd-supply-chain-arxiv-2506.06478.md         →   9
   06-stride-cloud-security-cisa-aligned.md                →  12
   07-astride-agentic-ai-arxiv-2512.04785.md               →   8
   08-stride-microservices-distributed-architectures.md    →  12
   09-isadm-stride-attack-d3fend-arxiv-2512.18751.md       →   9
   10-threatmodeling-llm-banking-arxiv-2411.17058.md       →   8
   11-aegisshield-democratizing-threat-modeling-arxiv-2509.10482.md → 

In [28]:
"""
=============================================================
CELL 6: Exportar chunks para JSON (backup / debug)
=============================================================
"""

output_path = PROJECT_ROOT / "RAG" / "chunks.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)

print(f"✅ Chunks exportados: {output_path}")
print(f"   Tamanho: {output_path.stat().st_size / 1024:.1f} KB")

✅ Chunks exportados: /Users/pedromarquardt/dev/estudos/tech-challenger/archguard-ai/ArchGuard-AI/RAG/chunks.json
   Tamanho: 228.4 KB


## Fase 2 — Embedding + Ingestão no ChromaDB

Requisitos:
1. `docker compose up -d` (ChromaDB rodando na porta 8000)
2. LM Studio rodando com `nomic-embed-text-v1.5` carregado (endpoint `/v1/embeddings`)
3. `.env` configurado com credenciais

In [29]:
!pip install python-dotenv openai chromadb -q


[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [30]:
"""
=============================================================
CELL 7: Carregar variáveis de ambiente
=============================================================
"""

from dotenv import load_dotenv

env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path)

CHROMA_HOST = os.getenv("CHROMA_HOST", "http://localhost")
CHROMA_PORT = int(os.getenv("CHROMA_PORT", 8000))
CHROMA_TOKEN = os.getenv("CHROMA_TOKEN", "")
LMS_HOST = os.getenv("LMS_HOST", "http://localhost:1234")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-nomic-embed-text-v1.5")

print(f"✅ Variáveis carregadas de: {env_path}")
print(f"   ChromaDB:  {CHROMA_HOST}:{CHROMA_PORT}")
print(f"   Token:     {'***' + CHROMA_TOKEN[-4:] if len(CHROMA_TOKEN) > 4 else '⚠️ não configurado'}")
print(f"   LM Studio: {LMS_HOST}")
print(f"   Modelo:    {EMBEDDING_MODEL}")

✅ Variáveis carregadas de: /Users/pedromarquardt/dev/estudos/tech-challenger/archguard-ai/ArchGuard-AI/.env
   ChromaDB:  http://localhost:8000
   Token:     ***2025
   LM Studio: http://localhost:1234
   Modelo:    text-embedding-nomic-embed-text-v1.5


In [31]:
"""
=============================================================
CELL 8: Gerar Embeddings via LM Studio
=============================================================
"""
from openai import OpenAI
import time

client = OpenAI(base_url=f"{LMS_HOST}/v1", api_key="lm-studio")

BATCH_SIZE = 32
all_embeddings = []
total_tokens = 0
start_time = time.time()

print(f"🚀 Gerando embeddings para {len(all_chunks)} chunks...")
print(f"   Modelo: {EMBEDDING_MODEL}")
print(f"   Batch size: {BATCH_SIZE}\n")

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i:i + BATCH_SIZE]
    texts = [c["text"] for c in batch]
    
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    
    batch_embeddings = [item.embedding for item in response.data]
    all_embeddings.extend(batch_embeddings)
    
    # Acumular usage se disponível
    if hasattr(response, "usage") and response.usage:
        total_tokens += response.usage.total_tokens
    
    batch_num = i // BATCH_SIZE + 1
    total_batches = (len(all_chunks) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"   ✅ Batch {batch_num}/{total_batches}: {len(batch)} chunks embedados | dim={len(batch_embeddings[0])}")

elapsed = time.time() - start_time

print(f"\n{'='*60}")
print(f"📊 Resumo dos Embeddings:")
print(f"   Total chunks:      {len(all_embeddings)}")
print(f"   Dimensão:          {len(all_embeddings[0])}")
print(f"   Tokens estimados:  {total_tokens if total_tokens else 'N/A (LM Studio)'}")
print(f"   Tempo total:       {elapsed:.2f}s")
print(f"   Throughput:        {len(all_embeddings)/elapsed:.1f} chunks/s")

🚀 Gerando embeddings para 109 chunks...
   Modelo: text-embedding-nomic-embed-text-v1.5
   Batch size: 32



   ✅ Batch 1/4: 32 chunks embedados | dim=768
   ✅ Batch 2/4: 32 chunks embedados | dim=768
   ✅ Batch 3/4: 32 chunks embedados | dim=768
   ✅ Batch 4/4: 13 chunks embedados | dim=768

📊 Resumo dos Embeddings:
   Total chunks:      109
   Dimensão:          768
   Tokens estimados:  N/A (LM Studio)
   Tempo total:       4.70s
   Throughput:        23.2 chunks/s


In [32]:
"""
=============================================================
CELL 9: Inserção no ChromaDB
=============================================================
"""
import chromadb
from chromadb.config import Settings

# Conectar ao ChromaDB com Token Auth correto
chroma_client = chromadb.HttpClient(
    host=CHROMA_HOST.replace("http://", "").replace("https://", ""),
    port=CHROMA_PORT,
    settings=Settings(
        # O caminho correto do cliente possui o _authn
        chroma_client_auth_provider="chromadb.auth.token_authn.TokenAuthClientProvider",
        chroma_client_auth_credentials=CHROMA_TOKEN
    )
)

# Criar/obter collection
collection = chroma_client.get_or_create_collection(
    name="stride-rag",
    metadata={"hnsw:space": "cosine"}
)

print(f"✅ Conectado ao ChromaDB: {CHROMA_HOST}:{CHROMA_PORT}")
print(f"   Collection: '{collection.name}'")
print(f"   Documentos existentes: {collection.count()}")

# Inserir em batches
print(f"\n🚀 Inserindo {len(all_chunks)} chunks com embeddings...\n")

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch_chunks = all_chunks[i:i + BATCH_SIZE]
    batch_embeds = all_embeddings[i:i + BATCH_SIZE]
    
    # ChromaDB não aceita None em metadatas — substituir por string vazia
    clean_metadatas = []
    for c in batch_chunks:
        meta = {k: (v if v is not None else "") for k, v in c["metadata"].items()}
        clean_metadatas.append(meta)
    
    collection.upsert(
        ids=[c["id"] for c in batch_chunks],
        documents=[c["text"] for c in batch_chunks],
        embeddings=batch_embeds,
        metadatas=clean_metadatas
    )
    
    batch_num = i // BATCH_SIZE + 1
    total_batches = (len(all_chunks) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"   ✅ Batch {batch_num}/{total_batches}: {len(batch_chunks)} chunks inseridos")

print(f"\n{'='*60}")
print(f"✅ Ingestão completa!")
print(f"   Collection: '{collection.name}'")
print(f"   Total documentos: {collection.count()}")

✅ Conectado ao ChromaDB: http://localhost:8000
   Collection: 'stride-rag'
   Documentos existentes: 0

🚀 Inserindo 109 chunks com embeddings...

   ✅ Batch 1/4: 32 chunks inseridos
   ✅ Batch 2/4: 32 chunks inseridos
   ✅ Batch 3/4: 32 chunks inseridos
   ✅ Batch 4/4: 13 chunks inseridos

✅ Ingestão completa!
   Collection: 'stride-rag'
   Total documentos: 109


In [33]:
"""
=============================================================
CELL 10: Teste de Query — Validação do RAG
=============================================================
"""

test_query = "Como aplicar STRIDE em microsserviços?"

# Gerar embedding da query
query_response = client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=[test_query]
)
query_embedding = query_response.data[0].embedding

print(f"✅ Embedding da query gerado (dim={len(query_embedding)})")
# Buscar no ChromaDB
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(f"🔍 Query: \"{test_query}\"\n")
print(f"📄 Top 3 resultados:\n")
for i, (doc_id, doc, meta, dist) in enumerate(zip(
    results["ids"][0],
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
)):
    print(f"{'─'*60}")
    print(f"  [{i+1}] ID: {doc_id}")
    print(f"      Fonte: {meta.get('source_file', 'N/A')}")
    print(f"      Seção: {meta.get('section_title', 'N/A')}")
    print(f"      Distância (cosine): {dist:.4f}")
    print(f"      Preview: {doc[:150]}...")
    print(f"metadatas {meta}")

✅ Embedding da query gerado (dim=768)
🔍 Query: "Como aplicar STRIDE em microsserviços?"

📄 Top 3 resultados:

────────────────────────────────────────────────────────────
  [1] ID: doc_08_s02_sub01
      Fonte: 08-stride-microservices-distributed-architectures.md
      Seção: 2. Adaptação do STRIDE para Microsserviços
      Distância (cosine): 0.1851
      Preview: # STRIDE Aplicado a Microsserviços: Framework para Análise de Arquiteturas Distribuídas
## 2. Adaptação do STRIDE para Microsserviços
### 2.1 Análise ...
metadatas {'relevancia': 'Alta — framework para análise de arquiteturas de microsserviços que o ArchGuard-AI deve analisar', 'source_file': '08-stride-microservices-distributed-architectures.md', 'referências': '> - "Threat Modeling for APIs in microservices architectures" — WJARR-2022-0458', 'fonte': 'Síntese de literatura acadêmica: Kanani & Dhenia (WJARR, 2022); IJNRD (2023); Nanjing University ACM (2022)', 'section_title': '2. Adaptação do STRIDE para Microsserviços', '

In [34]:
"""
=============================================================
CELL 11: Summary — Metadatas da Collection
=============================================================
"""
from collections import defaultdict

# Listar collections disponíveis
collections = chroma_client.list_collections()
print(f"📋 Collections disponíveis: {[c.name for c in collections]}\n")

if not collections:
    print("⚠️  Nenhuma collection encontrada. Execute as cells 4→9 para ingerir os dados.")
else:
    col = chroma_client.get_collection(name="stride-rag")
    all_data = col.get(include=["metadatas"])

    print(f"📊 Collection: '{col.name}'")
    print(f"   Total documentos: {len(all_data['ids'])}\n")

    # Coletar todas as chaves e valores únicos
    meta_summary = defaultdict(set)

    for meta in all_data["metadatas"]:
        for key, value in meta.items():
            if value and value != "":
                meta_summary[key].add(str(value))

    print(f"🔑 Metadata keys ({len(meta_summary)}):\n")
    for key in sorted(meta_summary.keys()):
        values = sorted(meta_summary[key])
        print(f"  {key}")
        if len(values) <= 15:
            for v in values:
                display = v[:80] + "..." if len(v) > 80 else v
                print(f"    • {display}")
        else:
            for v in values[:5]:
                display = v[:80] + "..." if len(v) > 80 else v
                print(f"    • {display}")
            print(f"    ... +{len(values) - 5} valores")
        print()

📋 Collections disponíveis: ['stride-rag']



📊 Collection: 'stride-rag'
   Total documentos: 109

🔑 Metadata keys (15):

  ano
    • Dezembro de 2024
    • Dezembro de 2025
    • Junho de 2024
    • Junho de 2025
    • Maio de 2025
    • Maio de 2026
    • Novembro de 2024
    • Setembro de 2025

  autor
    • Matthew Grofsky

  autores
    • Khondokar Fida Hasan, Hasibul Hossain Shajeeb, Chathura Abeydeera, Benjamin Turn...
    • Não especificados na extração
    • Não especificados na extração (pré-publicação 2025)
    • Pesquisadores (pré-publicação, Dezembro 2024)
    • Pesquisadores da Carleton University (NGN Security Group)
    • Saba Pourhanifeh, AbdulAziz AbdulGhaffar, Ashraf Matrawy
    • Sowmiya Dhandapani
    • Tingmin Wu, Shuiqiao Yang, Shigang Liu, David Nguyen, Seung Jang, Alsharif Abuad...

  categoria_arxiv
    • cs.CR
    • cs.CR, cs.AI

  fonte
    • CISA (Cybersecurity and Infrastructure Security Agency) — Cloud Security Best Pr...
    • Microsoft (criação original), OWASP, Threat Modeling Manifesto
    • Sínt